In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import  ChatOpenAI

load_dotenv()

model = ChatOpenAI(
        model_name="qwen3-max",
        api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# print(model)

使用 LangGraph 访问大模型的方式

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
)

agent.invoke({ "messages": [
    {
        "role": "user",
        "content": "你好"
    }
]})

In [ ]:
for chunk in agent.stream({
    "messages": [
        {
            "role": "user",
            "content": "你是谁，能帮我解决什么问题吗？"
        }
    ],
}, stream_mode="messages"):
    print(chunk)
    print("\n")

 增加工具调用

In [ ]:
import datetime
from langchain.agents import create_agent

def get_current_time():
    """获取当前时间"""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

agent = create_agent(
    model=model,
    tools=[get_current_time],
    system_prompt="你是一个时间助手，能获取当前时间，并返回给用户"
)

agent.invoke({
    "messages": [
        {"role": "user", "content": "当前时间是什么？"}
    ]
})

In [ ]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools import tool
from langgraph.prebuilt.tool_node import ToolCallRequest


@tool("divide_tool", return_direct=True)
def divide(a: int, b: int) -> float:
    """计算两个整数的除法
    Args:
        a: 被除数
        b: 除数
    """
    if b == 0:
        raise ValueError("除数不能为0")
    return a / b


@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """把工具异常转成 ToolMessage，让模型能继续处理，而不是直接崩溃。"""
    try:
        return handler(request)
    except Exception as error:
        if isinstance(error, ZeroDivisionError):
            content = "除数不能为0"
        elif isinstance(error, ValueError):
            content = f"输入的参数错误: {error}"
        else:
            content = f"工具执行错误: {error}"
        return ToolMessage(
            content=content,
            tool_call_id=request.tool_call["id"],
        )


# create_agent 会自己创建 ToolNode，不要把 ToolNode 放进 tools=
# 工具错误处理应通过 middleware=[...] 传入
agent_with_handle_tool_error = create_agent(
    model=model,
    tools=[divide],
    middleware=[handle_tool_errors],
    system_prompt="你是一个计算器，能计算两个整数的除法",
)

result = agent_with_handle_tool_error.invoke(
    {
        "messages": [
            {"role": "user", "content": "计算10除以5"},
        ]
    }
)

print(result['messages'][-1].content)